# 01 — Load Structure and Extract Residues

In this step, we load the 3D structure of hen egg-white lysozyme (1LYZ) and build a residue-level representation of the protein.

This includes:
- parsing the PDB structure
- extracting chain A
- converting residues to 1-letter sequence
- building a structured residue table

This table will be used later for mutation generation and stability analysis.

***Imports***

In [1]:
from pathlib import Path
from Bio.PDB import PDBParser
from Bio.PDB.Polypeptide import is_aa
from Bio.SeqUtils import seq1
import pandas as pd

In [2]:
from Bio.PDB.Polypeptide import is_aa
#important to keep only real amino-acid residues(skip water, hetero compounds, non-standard residues)

***Load structure***

In [3]:
pdb_path = Path("../data/input/1LYZ.pdb").resolve()

parser = PDBParser(QUIET=True)
structure = parser.get_structure("1LYZ", str(pdb_path))

model = structure[0]
chain = model["A"]

print("Structure loaded successfully")
print("PDB path:", pdb_path)
print("Chain ID:", chain.id)

Structure loaded successfully
PDB path: /Users/vaibhav.mh/protein_repos/mutation-effect-prediction/data/input/1LYZ.pdb
Chain ID: A


******Extract only standard amino acids******

In [6]:
residue_rows = []

for residue in chain:
    if not is_aa(residue, standard=True):
        continue

    hetflag, resseq, icode = residue.id
    resname = residue.get_resname()
    aa = seq1(resname)

    residue_rows.append({
        "chain": chain.id,
        "position": resseq,
        "insertion_code": icode.strip() if isinstance(icode, str) else "",
        "resname_3": resname,
        "resname_1": aa
    })

residue_df = pd.DataFrame(residue_rows)
residue_df.head(150)

,chain,position,insertion_code,resname_3,resname_1
0,A,1,,LYS,K
1,A,2,,VAL,V
2,A,3,,PHE,F
3,A,4,,GLY,G
4,A,5,,ARG,R
...,...,...,...,...,...
124,A,125,,ARG,R
125,A,126,,GLY,G
126,A,127,,CYS,C
127,A,128,,ARG,R


***Check residue count***

In [7]:
print("Total amino-acid residues:", len(residue_df))

Total amino-acid residues: 129


***Build the sequence again***

In [8]:
sequence = "".join(residue_df["resname_1"])
print("Sequence:")
print(sequence)
print("Sequence length:", len(sequence))

Sequence:
KVFGRCELAAAMKRHGLDNYRGYSLGNWVCAAKFESNFNTQATNRNTDGSTDYGILQINSRWWCNDGRTPGSRNLCNIPCSALLSSDITASVNCAKKIVSDGNGMNAWVAWRNRCKGTDVQAWIRGCRL
Sequence length: 129


***Save the cleaned residue table***

In [9]:
output_path = Path("../data/processed/residue_table.csv")
residue_df.to_csv(output_path, index=False)

print(f"Saved cleaned residue table to: {output_path}")

Saved cleaned residue table to: ../data/processed/residue_table.csv
